# ELITE vs HighCap v2 — clean rebuild

Fully self-contained. Each cell computes what it needs from scratch — no fragile inter-cell dependencies.

**Sections:**
1. Config
2. Load panel (both methods)
3. Run ELITE for all years
4. Compute HighCap for all years
5. Side-by-side picks per year with realized returns
6. Current year (2025) deep dive

## Strategy definitions

Every strategy takes the same starting universe (Section 2's `uni`, filtered by `POP_MIN`/`POP_MAX`/`HHI_MAX`) and returns a top-N list of counties with a per-state cap of `MAX_PER_STATE`. What differs is *how* the top-N is chosen.

---

### HC v1 — Raw cap-rate ranking

**Sort the universe by `cap_rate_proxy` descending. Take top-N with state cap. No filters.**

```
universe → sort(cap_rate_proxy desc) → apply state cap → top-N
```

Thesis: yield alone. Highest gross-rent-to-price ratio is the entry signal. Simplest possible model. Every other HC variant is a modification of this. Baseline against which we measure model complexity.

**Blind spot**: Rust Belt yield traps (Wayne MI, Cuyahoga OH, Philadelphia) — high yield forever because the market is right about the risk.

**Defined in**: Cell 8 — `highcap_picks_for(year)`

---

### HC v2 — Cap ranking + fundamental growth gates

**Same cap ranking as v1, but candidates must first pass hard filters on fundamentals. Ties broken by cap-rate compression momentum.**

Hard gates (NaN-permissive: missing data doesn't disqualify — only *known-bad* values do):

| Gate | Rule |
| --- | --- |
| `pop_yoy_pct` | ≥ 0 (population not shrinking) |
| `unemployment_rate` | ≤ 6.5% (labor market healthy) |
| `rental_vacancy_rate` | ≤ 12% (actual rental demand) |
| `renter_share` | ≥ 20% (meaningful renter base) |
| `any_disqualifier` | FALSE (mart's built-in aging-trap / sector-concentration / HPI-runup flags) |

**Momentum tiebreak** (`MOMENTUM_TIEBREAK = True`): within qualified pool, prefer counties with negative `mf_cap_rate_change_3yr` (cap-compressing = valuation tailwind).

```
universe → apply gates → sort(cap desc, mf_cap_change asc) → state cap → top-N
```

Thesis: high yield only counts if fundamentals aren't rotting. Empirically the gates barely bind in 2015-2019 (96%+ pass rate → picks identical to v1). They matter most in 2020-2022 (pass rate drops to 47-73%).

**Defined in**: Cell 28 — `highcap_v2_picks_for(year)`

---

### HC v3 — Cap ranking + "tops-guard" exclusion gates

**Same cap ranking as v1, but filter out markets showing signs of being at a cyclical top.**

Different philosophy from v2: instead of screening for structural weakness (which turned out to hurt), v3 screens for markets that are already frothy or in early recession.

Hard gates (NaN-permissive):

| Gate | Rule | What it excludes |
| --- | --- | --- |
| `hpi_trailing_24mo` | < 30% | Markets that already ran up 30%+ over 2 years |
| `permits_intensity_ratio` | < 1.5 | Oversupplied markets (current permits vs 10yr baseline) |
| `unemployment_rate_yoy_chg` | < 1.0 pp | Markets with unemployment spiking (recession warning) |
| `any_disqualifier` | FALSE | Mart's built-in structural filters |

```
universe → apply tops-guards → sort(cap desc) → state cap → top-N
```

Thesis: keep the "distressed high-cap mean-reversion" plays, but exclude "high-cap because we already ran the market up and it's about to correct" plays.

**Notable behavior**: at cyclical inflection points (e.g. 2020 with COVID-driven unemployment spikes), the guards can wipe out the entire eligible pool — v3 returns zero picks that year.

**Defined in**: Cell 34 — `highcap_v3_picks_for(year)`

---

### HC v4 — Cap shortlist + short-horizon rent momentum re-rank

**Two-stage sort: shortlist the top-K by cap, then re-rank inside that shortlist by trailing 12-month rent growth.**

```
universe → nlargest(cap_rate_proxy, K = TOP_N × 3) → sort(zori_yoy desc) → state cap → top-N
```

With `TOP_N = 5`, the shortlist is 15 counties. Inside that pool, cap-rate ordering is discarded and replaced by `zori_yoy` (trailing 12mo ZORI rent growth).

Thesis: don't buy the *highest*-cap markets — buy from among high-cap markets, the ones **already showing rent traction**. Combines yield entry with momentum confirmation.

**Key mechanic**: cap defines *eligibility* (top 15 only), rent momentum defines *ordering*. A #17-by-cap county with monster rent growth is ineligible; a #12-by-cap county with strong rent growth can beat a #2-by-cap county with weak rent growth.

**Defined in**: Cell 40 — `highcap_v4_picks_for(year, momentum_col='zori_yoy', shortlist_mult=3)`

---

### Cheat-sheet

| | Ranking key | Filter | Philosophy |
| --- | --- | --- | --- |
| **v1** | `cap_rate_proxy` desc | none | "yield only" |
| **v2** | `cap_rate_proxy` desc | growth gates | "yield + not broken" |
| **v3** | `cap_rate_proxy` desc | tops-guard gates | "yield + not frothy" |
| **v4** | `zori_yoy` desc (within top-15 cap shortlist) | none | "yield + rent trending up" |

For contrast, **ELITE** (Section 3) is fundamentally different — no cap-rate ranking. It identifies past top-25% winners by realized 3-yr total return, then ranks current candidates by *feature similarity* to those winners (cosine on ~90 z-scored features). See `backtest.analog_search.walk_forward`.

## 1. Config

In [89]:
from IPython.display import display, HTML
display(HTML("""
<style>
.jp-OutputArea-output { max-height: none !important; overflow: visible !important; }
.jp-OutputArea.jp-Cell-outputArea { max-height: none !important; }
div.output_scroll { height: auto !important; max-height: none !important; }
.output_wrapper .output { max-height: none !important; overflow: visible !important; }
</style>
"""))

import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '/Users/premputtegowda/reps/reps-tracker/emerging_re')

import duckdb
import numpy as np
import pandas as pd

WAREHOUSE = '/Users/premputtegowda/reps/reps-tracker/emerging_re/data/warehouse/emerging_re.duckdb'

# Edit these
POP_MIN = 300_000
POP_MAX = 500_000
HHI_MAX = 0.10
TOP_N = 5
MAX_PER_STATE = 2
ELITE_TOP_PCT = 0.25
HORIZON_YEARS = 3

pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_colwidth', 60)

print('Config loaded')
print('POP band:', POP_MIN, 'to', POP_MAX)
print('HHI ceiling:', HHI_MAX)
print('TOP_N:', TOP_N)
print('ELITE_TOP_PCT:', ELITE_TOP_PCT)

Config loaded
POP band: 300000 to 500000
HHI ceiling: 0.1
TOP_N: 5
ELITE_TOP_PCT: 0.25


## 2. Load universe (with realized returns) — once for all years

In [90]:
# Load the full universe across ALL years, plus exit caps for TR computation
sql_uni = '''
    SELECT fips, county_name, state_name, pop_latest,
           EXTRACT(YEAR FROM period_end_date)::INT AS year,
           cap_rate_proxy, mf_cap_rate_estimated, employment_diversity_hhi,
           zori_forward_return_36mo
    FROM main_marts.mart_county_features
    WHERE EXTRACT(MONTH FROM period_end_date) = 12
      AND pop_latest > {pmin} AND pop_latest <= {pmax}
      AND employment_diversity_hhi IS NOT NULL
      AND employment_diversity_hhi <= {hhi}
'''.format(pmin=POP_MIN, pmax=POP_MAX, hhi=HHI_MAX)

con = duckdb.connect(WAREHOUSE, read_only=True)
uni = con.execute(sql_uni).df()
con.close()

# Attach exit cap 3 years later (self-join)
exit_lookup = uni[['fips','year','cap_rate_proxy']].copy()
exit_lookup['year'] = exit_lookup['year'] - HORIZON_YEARS   # shift year backwards so join brings +3yr forward
exit_lookup = exit_lookup.rename(columns={'cap_rate_proxy': 'cap_exit'})

uni = uni.merge(exit_lookup, on=['fips','year'], how='left')

uni['tr_zori36'] = (1 + uni['zori_forward_return_36mo']) * (uni['cap_rate_proxy'] / uni['cap_exit']) - 1

print('Universe rows:', len(uni))
print('Years:', sorted(uni['year'].unique()))
print('Universe size per year (last 12):')
print(uni.groupby('year').size().tail(12).to_string())

Universe rows: 2309
Years: [np.int32(2000), np.int32(2001), np.int32(2002), np.int32(2003), np.int32(2004), np.int32(2005), np.int32(2006), np.int32(2007), np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
Universe size per year (last 12):
year
2014    91
2015    89
2016    89
2017    89
2018    89
2019    89
2020    89
2021    89
2022    88
2023    87
2024    88
2025    87


## 3. Run ELITE for all test years

In [91]:
from backtest.analog_search import (
    load_tertiary_panel, load_mf_context, build_acs_target,
    build_feature_matrix, zscore_within_cohort, walk_forward,
    TARGET_COL_TOTAL_RETURN,
)

# Load panel and features (uses the same POP/HHI knobs as Section 2)
panel = load_tertiary_panel(pop_min=POP_MIN, pop_max=POP_MAX, hhi_max=HHI_MAX)
panel = panel.merge(build_acs_target(horizon_years=HORIZON_YEARS), on=['fips','period_end_date'], how='left')
panel = panel.merge(load_mf_context(), on='fips', how='left')

panel_fe, feature_cols = build_feature_matrix(panel)
panel_z = zscore_within_cohort(panel_fe, feature_cols)

print('Panel:', len(panel), 'rows,', len(feature_cols), 'features')

# Run ELITE walk-forward (all test years, strict PIT, ELITE top 25%)
elite_results = walk_forward(
    panel_z, panel_fe, feature_cols,
    target_col=TARGET_COL_TOTAL_RETURN,
    strict_pit_targets=True,
    top_n=TOP_N,
    max_per_state=MAX_PER_STATE,
    elite_top_pct=ELITE_TOP_PCT,
)

print('ELITE completed:', len(elite_results), 'iterations')

Panel: 2309 rows, 91 features

Walk-forward schedule (21 iterations, min_train_years=5, target=target_total_return_36mo_demeaned, top_pct=0.25, top_n=5, max_per_state=2, strict_pit=True):
  SKIP test=2005   train_cohorts= 0   span=(none)
  SKIP test=2006   train_cohorts= 0   span=(none)
  SKIP test=2007   train_cohorts= 0   span=(none)
  SKIP test=2008   train_cohorts= 0   span=(none)
  SKIP test=2009   train_cohorts= 0   span=(none)
  SKIP test=2010   train_cohorts= 0   span=(none)
  SKIP test=2011   train_cohorts= 1   span=2008-2008
  SKIP test=2012   train_cohorts= 2   span=2008-2009
  SKIP test=2013   train_cohorts= 3   span=2008-2010
  SKIP test=2014   train_cohorts= 4   span=2008-2011
  test=2015   train_cohorts= 5   span=2008-2012
  test=2016   train_cohorts= 6   span=2008-2013
  test=2017   train_cohorts= 7   span=2008-2014
  test=2018   train_cohorts= 8   span=2008-2015
  test=2019   train_cohorts= 9   span=2008-2016
  test=2020   train_cohorts=10   span=2008-2017
  test=2021 

## 4. Compute HighCap picks for all years (simple heuristic — no model)

In [92]:
def highcap_picks_for(year):
    '''Top-N counties by cap_rate_proxy in the given year, with state cap.'''
    c = uni[uni['year'] == year].dropna(subset=['cap_rate_proxy']).sort_values('cap_rate_proxy', ascending=False)
    picked = []
    state_counts = {}
    for _, r in c.iterrows():
        s = r['state_name']
        if state_counts.get(s, 0) >= MAX_PER_STATE:
            continue
        picked.append(r)
        state_counts[s] = state_counts.get(s, 0) + 1
        if len(picked) >= TOP_N:
            break
    return pd.DataFrame(picked) if picked else pd.DataFrame()

# Sample check
hc_2020 = highcap_picks_for(2020)
print('HighCap 2020 picks (', len(hc_2020), '):')
print(hc_2020[['county_name','state_name','cap_rate_proxy','tr_zori36']].to_string(index=False))

HighCap 2020 picks ( 5 ):
      county_name     state_name  cap_rate_proxy  tr_zori36
Cumberland County North Carolina        0.082955   0.558141
    Nueces County          Texas        0.081170   0.343180
   Luzerne County   Pennsylvania        0.077327   0.470680
   Lubbock County          Texas        0.075362   0.362920
     Lucas County           Ohio        0.073140   0.329277


## 5. Per-year side-by-side picks

Both methods for every test year (2015-2025). ⭐ = consensus (in both methods).

In [93]:
summary_rows = []
year_tr_lookup = uni[["fips","year","tr_zori36"]].dropna(subset=["tr_zori36"])

for r in elite_results:
    yr = r.test_year
    if yr < 2015:
        continue

    elite_picks = r.top5[["fips","county_name","state_name"]].copy()
    hc_picks = highcap_picks_for(yr)
    if hc_picks.empty:
        continue

    hc_fips = list(hc_picks["fips"])
    el_fips = list(elite_picks["fips"])
    inter = set(hc_fips) & set(el_fips)

    # Realized TR per method — pull from uni directly (avoids merge collisions)
    year_tr = year_tr_lookup[year_tr_lookup["year"] == yr][["fips","tr_zori36"]]
    if len(year_tr) > 0:
        tr_map = dict(zip(year_tr["fips"], year_tr["tr_zori36"]))
        hc_trs = [tr_map.get(f) for f in hc_fips]
        el_trs = [tr_map.get(f) for f in el_fips]
        hc_tr = np.nanmean([t for t in hc_trs if t is not None]) * 100 if any(t is not None for t in hc_trs) else np.nan
        el_tr = np.nanmean([t for t in el_trs if t is not None]) * 100 if any(t is not None for t in el_trs) else np.nan
        uni_tr = year_tr["tr_zori36"].mean() * 100
        actual_top_fips = set(year_tr.nlargest(TOP_N, "tr_zori36")["fips"])
    else:
        hc_tr, el_tr, uni_tr = np.nan, np.nan, np.nan
        actual_top_fips = set()

    print("")
    print("=" * 100)
    hc_str = "n/a" if pd.isna(hc_tr) else f"{hc_tr:+.2f}%"
    el_str = "n/a" if pd.isna(el_tr) else f"{el_tr:+.2f}%"
    uni_str = "n/a" if pd.isna(uni_tr) else f"{uni_tr:+.2f}%"
    print(f"YEAR {yr}   universe n={len(uni[uni['year']==yr])}   HC TR={hc_str}   EL TR={el_str}   Universe={uni_str}")
    print("=" * 100)
    print(f"{'#':<3}  {'HighCap':<32}  {'ELITE':<32}")
    for i in range(TOP_N):
        hc_r = hc_picks.iloc[i] if i < len(hc_picks) else None
        el_r = elite_picks.iloc[i] if i < len(elite_picks) else None
        def fmt(r):
            if r is None: return "".ljust(32)
            marks = ""
            if r["fips"] in inter: marks += "*"
            if r["fips"] in actual_top_fips: marks += "+"
            name = r["county_name"].replace(" County","") + ", " + r["state_name"][:2]
            return (marks + name)[:32].ljust(32)
        print(f"{i+1:<3}  {fmt(hc_r)}  {fmt(el_r)}")

    if inter:
        inter_names = sorted(set(uni[(uni["year"]==yr) & (uni["fips"].isin(inter))]["county_name"].str.replace(" County","")))
        print("  * Consensus:", ", ".join(inter_names))

    summary_rows.append({
        "year": yr,
        "n_universe": len(uni[uni["year"]==yr]),
        "HC_TR": hc_tr,
        "EL_TR": el_tr,
        "uni_TR": uni_tr,
        "consensus_count": len(inter),
    })

print()
print("Legend: * = consensus pick (in both methods), + = in actual top-N by realized TR")



YEAR 2015   universe n=89   HC TR=+32.04%   EL TR=+20.68%   Universe=+20.77%
#    HighCap                           ELITE                           
1    +St. Lucie, Fl                    Boulder, Co                     
2    Genesee, Mi                       Hamilton, In                    
3    Nueces, Te                        Douglas, Co                     
4    Marion, Fl                        Benton, Ar                      
5    Lubbock, Te                       St. Johns, Fl                   

YEAR 2016   universe n=89   HC TR=+21.40%   EL TR=+28.60%   Universe=+19.27%
#    HighCap                           ELITE                           
1    Nueces, Te                        Boulder, Co                     
2    Genesee, Mi                       Washtenaw, Mi                   
3    Escambia, Fl                      +St. Johns, Fl                  
4    Cumberland, No                    Rutherford, Te                  
5    Lubbock, Te                       Larimer, Co  

In [94]:
# Summary table
summary_df = pd.DataFrame(summary_rows)
summary_df['HC_edge'] = summary_df['HC_TR'] - summary_df['uni_TR']
summary_df['EL_edge'] = summary_df['EL_TR'] - summary_df['uni_TR']

print('=' * 100)
print('SUMMARY across all years')
print('=' * 100)
print(summary_df.to_string(index=False, float_format=lambda x: f'{x:+.2f}' if pd.notna(x) else 'n/a'))

obs = summary_df.dropna(subset=['HC_TR','EL_TR'])
if len(obs):
    print()
    print('Aggregate over', len(obs), 'observable cohorts:')
    print('  HighCap avg TR:', round(obs['HC_TR'].mean(), 2), '%   edge:', round(obs['HC_edge'].mean(), 2), 'pp')
    print('  ELITE   avg TR:', round(obs['EL_TR'].mean(), 2), '%   edge:', round(obs['EL_edge'].mean(), 2), 'pp')
    print('  Avg consensus picks per year:', round(summary_df['consensus_count'].mean(), 1))

SUMMARY across all years
 year  n_universe  HC_TR  EL_TR  uni_TR  consensus_count  HC_edge  EL_edge
 2015          89 +32.04 +20.68  +20.77                0   +11.27    -0.09
 2016          89 +21.40 +28.60  +19.27                0    +2.13    +9.33
 2017          89 +32.72 +24.00  +19.89                0   +12.82    +4.11
 2018          89 +46.18 +30.51  +29.54                0   +16.64    +0.98
 2019          89 +48.60 +38.18  +40.71                0    +7.89    -2.53
 2020          89 +41.28 +56.43  +43.93                0    -2.65   +12.50
 2021          89 +29.80 +29.79  +26.70                1    +3.09    +3.09
 2022          88 +22.36  +1.13  +14.66                0    +7.70   -13.53
 2023          87    NaN    NaN     NaN                1      NaN      NaN
 2024          88    NaN    NaN     NaN                0      NaN      NaN
 2025          87    NaN    NaN     NaN                0      NaN      NaN

Aggregate over 8 observable cohorts:
  HighCap avg TR: 34.3 %   edge: 7.36

## 6. Current year (2025) — deep dive

In [95]:
CURRENT_YEAR = 2025

elite_2025 = next((r for r in elite_results if r.test_year == CURRENT_YEAR), None)
if elite_2025 is None:
    print('No ELITE result for', CURRENT_YEAR)
else:
    el_picks = elite_2025.top5[['fips','county_name','state_name']].copy()
    hc_picks = highcap_picks_for(CURRENT_YEAR)

    # Attach cap rates from mart
    caps = uni[uni['year'] == CURRENT_YEAR][['fips','cap_rate_proxy','mf_cap_rate_estimated','employment_diversity_hhi','pop_latest']]
    el_picks = el_picks.merge(caps, on='fips', how='left')

    inter = set(hc_picks['fips']) & set(el_picks['fips'])

    print('=' * 130)
    print(CURRENT_YEAR, 'top-', TOP_N, '  (pop', POP_MIN, 'to', POP_MAX, ', HHI <=', HHI_MAX, ')')
    print('=' * 130)
    print(f"{'#':<3}  {'HighCap':<38}  {'cap':>7}  |  {'ELITE':<38}  {'cap':>7}")
    for i in range(TOP_N):
        hc_r = hc_picks.iloc[i] if i < len(hc_picks) else None
        el_r = el_picks.iloc[i] if i < len(el_picks) else None
        def fmt(r):
            if r is None: return ' '*46
            mark = '*' if r['fips'] in inter else ' '
            name = mark + r['county_name'].replace(' County','') + ', ' + r['state_name']
            cap = f"{r['cap_rate_proxy']*100:.2f}%" if pd.notna(r['cap_rate_proxy']) else 'n/a'
            return f"{name:<38}  {cap:>7}"
        print(f"{i+1:<3}  {fmt(hc_r)}  |  {fmt(el_r)}")

    print()
    print('Consensus picks:', len(inter), 'of', TOP_N)
    if inter:
        for f in inter:
            r = uni[(uni['year']==CURRENT_YEAR) & (uni['fips']==f)].iloc[0]
            mf = r['mf_cap_rate_estimated']
            mf_str = f"{mf*100:.2f}%" if pd.notna(mf) else 'n/a'
            print(f"  ⭐ {r['county_name']}, {r['state_name']}  pop={int(r['pop_latest'])}  cap_ACS={r['cap_rate_proxy']*100:.2f}%  cap_MF={mf_str}  HHI={r['employment_diversity_hhi']:.4f}')")
    else:
        print('  (none — the two methods are orthogonal for this cohort)')

2025 top- 5   (pop 300000 to 500000 , HHI <= 0.1 )
#    HighCap                                     cap  |  ELITE                                       cap
1     Cumberland, North Carolina               7.24%  |   St. Johns, Florida                       4.53%
2     Marion, Florida                          6.63%  |   Benton, Arkansas                         4.23%
3     Nueces, Texas                            6.60%  |   Chesterfield, Virginia                   5.20%
4     Luzerne, Pennsylvania                    6.55%  |   Henrico, Virginia                        5.20%
5     Lubbock, Texas                           6.52%  |   Hays, Texas                              4.49%

Consensus picks: 0 of 5
  (none — the two methods are orthogonal for this cohort)


## 7. Per-pick performance table

For each cohort year, show every pick with its individual **entry cap**, **exit cap**, **cap Δ (bps)**, **ZORI 36mo rent growth**, and **Total Return (rent × cap compression)**. Method column shows HC / EL / Both.

In [96]:
# For each test year and each pick, produce a row with pick details + realized components

def build_pick_perf(elite_results, uni, top_n, min_year=2015):
    # Pre-compute per-fips-per-year lookup with cap_entry, cap_exit, zori36
    lookup = uni[["fips","year","cap_rate_proxy","cap_exit","zori_forward_return_36mo","tr_zori36","county_name","state_name","pop_latest"]].copy()
    lookup = lookup.rename(columns={"cap_rate_proxy": "cap_entry"})

    rows = []
    for r in elite_results:
        yr = r.test_year
        if yr < min_year:
            continue

        el_fips = list(r.top5["fips"])
        hc_picks_yr = highcap_picks_for(yr)
        hc_fips = list(hc_picks_yr["fips"]) if not hc_picks_yr.empty else []

        # Union: every unique pick across both methods
        all_fips = list(dict.fromkeys(hc_fips + el_fips))  # preserves order, deduplicates

        for f in all_fips:
            m = lookup[(lookup["year"] == yr) & (lookup["fips"] == f)]
            if m.empty:
                continue
            m = m.iloc[0]

            method_marks = []
            if f in hc_fips: method_marks.append("HC")
            if f in el_fips: method_marks.append("EL")
            method_str = "+".join(method_marks)

            cap_e = m["cap_entry"]
            cap_x = m["cap_exit"]
            cap_d_bps = (cap_x - cap_e) * 10000 if pd.notna(cap_e) and pd.notna(cap_x) else np.nan
            zori36 = m["zori_forward_return_36mo"]
            tr36 = m["tr_zori36"]

            rows.append({
                "year": yr,
                "method": method_str,
                "county": m["county_name"].replace(" County",""),
                "state": m["state_name"][:2],
                "pop": int(m["pop_latest"]),
                "cap_entry_%": round(cap_e * 100, 2) if pd.notna(cap_e) else np.nan,
                "cap_exit_%": round(cap_x * 100, 2) if pd.notna(cap_x) else np.nan,
                "cap_delta_bps": round(cap_d_bps, 0) if pd.notna(cap_d_bps) else np.nan,
                "ZORI_36mo_%": round(zori36 * 100, 2) if pd.notna(zori36) else np.nan,
                "Total_TR_%": round(tr36 * 100, 2) if pd.notna(tr36) else np.nan,
            })
    return pd.DataFrame(rows)

perf_df = build_pick_perf(elite_results, uni, TOP_N)
print("Rows:", len(perf_df))
print("Columns:", list(perf_df.columns))

Rows: 108
Columns: ['year', 'method', 'county', 'state', 'pop', 'cap_entry_%', 'cap_exit_%', 'cap_delta_bps', 'ZORI_36mo_%', 'Total_TR_%']


### 7.1 Full table — all cohorts, all picks, sorted by year then method

In [97]:
# Show the full table year by year
for yr in sorted(perf_df["year"].unique()):
    sub = perf_df[perf_df["year"] == yr].sort_values(["method","cap_entry_%"], ascending=[True, False])
    print("")
    print("=" * 120)
    print("YEAR", yr, "  ({} picks across both methods)".format(len(sub)))
    print("=" * 120)
    print(sub[["method","county","state","pop","cap_entry_%","cap_exit_%","cap_delta_bps","ZORI_36mo_%","Total_TR_%"]].to_string(index=False))


YEAR 2015   (10 picks across both methods)
method    county state    pop  cap_entry_%  cap_exit_%  cap_delta_bps  ZORI_36mo_%  Total_TR_%
    EL    Benton    Ar 332554         6.36        5.86          -50.0        12.58       22.10
    EL St. Johns    Fl 346328         5.52        5.23          -29.0        12.53       18.77
    EL  Hamilton    In 387036         5.42        4.91          -51.0         6.47       17.51
    EL   Douglas    Co 399396         4.77        4.17          -60.0         7.20       22.54
    EL   Boulder    Co 328560         3.85        3.53          -33.0        12.11       22.48
    HC St. Lucie    Fl 402449         9.84        8.05         -179.0        19.83       46.46
    HC   Genesee    Mi 401093         9.76        7.96         -179.0        12.93       38.37
    HC    Nueces    Te 352992         9.35        8.35         -101.0         5.25       17.96
    HC    Marion    Fl 442660         8.81        7.49         -132.0        16.68       37.31
    HC

### 7.2 Summary — average per-pick performance grouped by method

In [98]:
# Aggregate per method label (HC-only / EL-only / Both)
grouped = perf_df.dropna(subset=["Total_TR_%"]).groupby("method").agg(
    n_picks=("Total_TR_%", "count"),
    avg_cap_entry=("cap_entry_%", "mean"),
    avg_cap_exit=("cap_exit_%", "mean"),
    avg_cap_delta_bps=("cap_delta_bps", "mean"),
    avg_ZORI_36mo=("ZORI_36mo_%", "mean"),
    avg_Total_TR=("Total_TR_%", "mean"),
).round(2)

print("=" * 90)
print("Per-method average performance (observable cohorts only)")
print("=" * 90)
print(grouped.to_string())

# Also per-year per-method
print()
print("=" * 90)
print("Per-year × method breakdown")
print("=" * 90)
yr_method = perf_df.dropna(subset=["Total_TR_%"]).groupby(["year","method"]).agg(
    n=("Total_TR_%", "count"),
    avg_TR=("Total_TR_%", "mean"),
    avg_ZORI=("ZORI_36mo_%", "mean"),
    avg_cap_d_bps=("cap_delta_bps", "mean"),
).round(2).reset_index()
print(yr_method.to_string(index=False))

Per-method average performance (observable cohorts only)
        n_picks  avg_cap_entry  avg_cap_exit  avg_cap_delta_bps  avg_ZORI_36mo  avg_Total_TR
method                                                                                      
EL           38           5.08          4.64             -43.66          16.76         27.54
HC           39           8.10          7.20             -89.46          18.87         34.06
HC+EL         1           8.30          6.87            -143.00          18.95         43.70

Per-year × method breakdown
 year method  n  avg_TR  avg_ZORI  avg_cap_d_bps
 2015     EL  5   20.68     10.18         -44.60
 2015     HC  5   32.04     11.87        -140.80
 2016     EL  5   28.60     12.89         -61.80
 2016     HC  5   21.40     11.43         -68.40
 2017     EL  5   24.00     11.99         -51.20
 2017     HC  5   32.72     15.74        -100.00
 2018     EL  5   30.51     25.39         -17.80
 2018     HC  5   46.18     30.34         -77.20
 2019   

### 7.3 Best and worst individual picks across all cohorts

In [99]:
# Best-N and worst-N picks by Total Return, ignoring not-yet-observable cohorts
obs_df = perf_df.dropna(subset=["Total_TR_%"])

print("=" * 90)
print("TOP 15 individual picks by realized Total Return")
print("=" * 90)
print(obs_df.nlargest(15, "Total_TR_%")[["year","method","county","state","cap_entry_%","cap_exit_%","cap_delta_bps","ZORI_36mo_%","Total_TR_%"]].to_string(index=False))

print()
print("=" * 90)
print("BOTTOM 15 individual picks by realized Total Return")
print("=" * 90)
print(obs_df.nsmallest(15, "Total_TR_%")[["year","method","county","state","cap_entry_%","cap_exit_%","cap_delta_bps","ZORI_36mo_%","Total_TR_%"]].to_string(index=False))

TOP 15 individual picks by realized Total Return
 year method     county state  cap_entry_%  cap_exit_%  cap_delta_bps  ZORI_36mo_%  Total_TR_%
 2020     EL  St. Lucie    Fl         6.35        4.67         -168.0        37.64       87.08
 2018     HC  St. Lucie    Fl         8.05        6.35         -170.0        44.73       83.55
 2019     HC   Escambia    Fl         7.68        6.19         -149.0        41.76       76.00
 2018     HC   Escambia    Fl         7.95        6.96          -99.0        45.10       65.68
 2020     HC Cumberland    No         8.30        7.15         -114.0        34.34       55.81
 2019     HC Cumberland    No         8.01        7.78          -23.0        46.11       50.44
 2017     HC  St. Lucie    Fl         8.04        6.35         -169.0        18.73       50.38
 2017     EL Rutherford    Te         6.91        5.31         -159.0        14.50       48.85
 2018     EL  St. Johns    Fl         5.23        4.73          -50.0        33.89       48.04
 

## 8. Top-5 picks per year — clean side-by-side with rent / cap / TR

Compact focus: just the top 5 picks per method per year, with individual pick performance.

In [100]:
# GUARD: define highcap_v4_picks_for if not already loaded (Section 12 dependency)
if "highcap_v4_picks_for" not in dir():
    print("  Loading HC v4 helper inline...")
    _con_v4_g = duckdb.connect(WAREHOUSE, read_only=True)
    _zori_yoy_df_g = _con_v4_g.execute("""
        SELECT fips, EXTRACT(YEAR FROM period_end_date)::INT AS year, zori_yoy, hpi_yoy
        FROM main_marts.mart_county_features
        WHERE EXTRACT(MONTH FROM period_end_date) = 12
    """).df()
    _con_v4_g.close()

    def highcap_v4_picks_for(year, momentum_col="zori_yoy", shortlist_mult=3):
        c = uni[uni["year"] == year].dropna(subset=["cap_rate_proxy"]).copy()
        c = c.merge(_zori_yoy_df_g[_zori_yoy_df_g["year"] == year][["fips","zori_yoy","hpi_yoy"]], on="fips", how="left")
        K = TOP_N * shortlist_mult
        shortlist = c.nlargest(K, "cap_rate_proxy").copy()
        shortlist = shortlist.sort_values(momentum_col, ascending=False, na_position="last")
        picked, sc = [], {}
        for _, r in shortlist.iterrows():
            if sc.get(r["state_name"], 0) >= MAX_PER_STATE: continue
            picked.append(r); sc[r["state_name"]] = sc.get(r["state_name"], 0) + 1
            if len(picked) >= TOP_N: break
        return pd.DataFrame(picked), len(shortlist)

TOP_5 = 5

# Load context: unemployment, vacancy, renter_share (all PIT-correct from mart)
con_c = duckdb.connect(WAREHOUSE, read_only=True)
ctx = con_c.execute("""
    SELECT fips,
           EXTRACT(YEAR FROM period_end_date)::INT AS year,
           unemployment_rate,
           rental_vacancy_rate,
           renter_share
    FROM main_marts.mart_county_features
    WHERE EXTRACT(MONTH FROM period_end_date) = 12
""").df()

# Raw PEP with as_of_date for strict-PIT pop YoY
pep_raw = con_c.execute("""
    SELECT fips,
           EXTRACT(YEAR FROM period_end_date)::INT AS ref_year,
           popestimate,
           as_of_date
    FROM main_staging.stg_census_pep
""").df()
con_c.close()

pep_raw["as_of_date"] = pd.to_datetime(pep_raw["as_of_date"])

def pit_pop_yoy(cohort_year):
    """PIT-strict pop YoY: use latest vintage available by cohort Dec-31."""
    cutoff = pd.Timestamp(f"{cohort_year}-12-31")
    df = pep_raw[pep_raw["as_of_date"] <= cutoff].copy()
    if df.empty:
        return pd.DataFrame(columns=["fips","pop_yoy_pct","pop_yoy_years"])
    df = df.sort_values(["fips","ref_year","as_of_date"]).drop_duplicates(subset=["fips","ref_year"], keep="last")
    out_rows = []
    for f, sub in df.groupby("fips"):
        sub = sub.sort_values("ref_year", ascending=False)
        if len(sub) < 2: continue
        r_latest = sub.iloc[0]
        r_prev = sub[sub["ref_year"] == r_latest["ref_year"] - 1]
        if r_prev.empty: continue
        r_prev = r_prev.iloc[0]
        yoy = (r_latest["popestimate"] - r_prev["popestimate"]) / r_prev["popestimate"] * 100
        out_rows.append({
            "fips": f,
            "pop_yoy_pct": yoy,
            "pop_yoy_years": f"{int(r_prev['ref_year'])}→{int(r_latest['ref_year'])}",
        })
    return pd.DataFrame(out_rows)

pop_yoy_cache = {}
for yr_test in range(2015, 2026):
    pop_yoy_cache[yr_test] = pit_pop_yoy(yr_test)

# Master enrichment lookup
enrich = uni[["fips","year","cap_rate_proxy","cap_exit","zori_forward_return_36mo","tr_zori36"]].copy()
enrich = enrich.rename(columns={"cap_rate_proxy":"cap_entry"})
enrich = enrich.merge(ctx[["fips","year","unemployment_rate","rental_vacancy_rate","renter_share"]], on=["fips","year"], how="left")

def fmt_ret(v, w=6):
    return f"{v*100:+{w}.2f}%" if pd.notna(v) else f"{'n/a':>{w+1}s}"
def fmt_frac_as_pct(v, w=5):
    return f"{v*100:{w}.2f}%" if pd.notna(v) else f"{'n/a':>{w+1}s}"
def fmt_already_pct(v, w=5):
    return f"{v:{w}.2f}%" if pd.notna(v) else f"{'n/a':>{w+1}s}"
def fmt_bps(entry, exit_):
    if pd.notna(entry) and pd.notna(exit_):
        return f"{(exit_ - entry)*10000:+6.0f}"
    return "   n/a"
def fmt_pop_yoy(v):
    return f"{v:+5.2f}%" if pd.notna(v) else "  n/a "

for r in elite_results:
    yr = r.test_year
    if yr < 2015:
        continue

    el_top5 = r.top5.head(TOP_5)[["fips","county_name","state_name"]].copy()
    hc_top5 = highcap_picks_for(yr).head(TOP_5)[["fips","county_name","state_name"]].copy() if not highcap_picks_for(yr).empty else pd.DataFrame()
    hc_v4_picks, _ = highcap_v4_picks_for(yr)
    hc_v4_top5 = hc_v4_picks.head(TOP_5)[["fips","county_name","state_name"]].copy() if not hc_v4_picks.empty else pd.DataFrame()

    pop_yoy_yr = pop_yoy_cache.get(yr, pd.DataFrame())

    def enrich_picks(picks):
        if picks.empty: return picks
        m = picks.merge(enrich[enrich["year"] == yr], on="fips", how="left")
        if not pop_yoy_yr.empty:
            m = m.merge(pop_yoy_yr, on="fips", how="left")
        else:
            m["pop_yoy_pct"] = None
            m["pop_yoy_years"] = None
        return m

    el_e = enrich_picks(el_top5)
    hc_e = enrich_picks(hc_top5)
    hc_v4_e = enrich_picks(hc_v4_top5)

    years_note = ""
    if not pop_yoy_yr.empty and "pop_yoy_years" in pop_yoy_yr.columns:
        most_common = pop_yoy_yr["pop_yoy_years"].mode()
        if len(most_common) > 0:
            years_note = f"   [pop YoY vintage available at {yr}-12-31: {most_common.iloc[0]}]"

    print("")
    print("=" * 165)
    print(f"YEAR {yr}   (universe n={len(uni[uni['year']==yr])}){years_note}")
    print("=" * 165)

    def print_block(title, picks):
        print(f"\n{title}")
        print(f"{'#':<3} {'Market':<30} {'cap_in':>7} {'cap_out':>7} {'Δbps':>6} {'ZORI36':>8} {'TR':>8}   {'unemp':>6} {'vacncy':>6} {'renter':>6} {'popYoY':>7}")
        print("-" * 145)
        for i in range(TOP_5):
            r = picks.iloc[i] if i < len(picks) else None
            if r is None:
                print(f"#{i+1:<2}")
                continue
            name = f"{r['county_name'].replace(' County','')}, {r['state_name'][:2]}"
            print(f"#{i+1:<2} {name:<30} {fmt_ret(r['cap_entry'])} {fmt_ret(r['cap_exit'])} {fmt_bps(r['cap_entry'], r['cap_exit']):>6} {fmt_ret(r['zori_forward_return_36mo'])} {fmt_ret(r['tr_zori36'])}   {fmt_already_pct(r['unemployment_rate']):>6} {fmt_frac_as_pct(r['rental_vacancy_rate']):>6} {fmt_frac_as_pct(r['renter_share']):>6} {fmt_pop_yoy(r['pop_yoy_pct']):>7}")

    print_block("HighCap v1 (raw cap) top-5:", hc_e)
    print_block("HighCap v4 (cap + momentum) top-5:", hc_v4_e)
    print_block("ELITE top-5:", el_e)

    def print_avg(label, picks):
        if picks.empty or picks["tr_zori36"].isna().all(): return
        avg_tr = picks["tr_zori36"].mean(skipna=True)*100
        avg_zori = picks["zori_forward_return_36mo"].mean(skipna=True)*100
        avg_cap_d = (picks["cap_exit"] - picks["cap_entry"]).mean(skipna=True)*10000
        print(f"{label}: ZORI36={avg_zori:+.2f}%   cap Δ={avg_cap_d:+.0f}bps   TR={avg_tr:+.2f}%")

    print()
    print_avg("HC v1 top-5 avg", hc_e)
    print_avg("HC v4 top-5 avg", hc_v4_e)
    print_avg("EL    top-5 avg", el_e)



YEAR 2015   (universe n=89)

HighCap v1 (raw cap) top-5:
#   Market                          cap_in cap_out   Δbps   ZORI36       TR    unemp vacncy renter  popYoY
-------------------------------------------------------------------------------------------------------------------------------------------------
#1  St. Lucie, Fl                   +9.84%  +8.05%   -179 +19.83% +46.46%    5.60% 10.57% 26.68%    n/a 
#2  Genesee, Mi                     +9.76%  +7.96%   -179 +12.93% +38.37%    5.20%  7.82% 30.61%    n/a 
#3  Nueces, Te                      +9.35%  +8.35%   -101  +5.25% +17.96%    5.10%  6.38% 41.89%    n/a 
#4  Marion, Fl                      +8.81%  +7.49%   -132 +16.68% +37.31%    5.90% 10.49% 24.10%    n/a 
#5  Lubbock, Te                     +8.81%  +7.68%   -113  +4.65% +20.10%    3.00%  7.23% 42.76%    n/a 

HighCap v4 (cap + momentum) top-5:
#   Market                          cap_in cap_out   Δbps   ZORI36       TR    unemp vacncy renter  popYoY
---------------------

In [102]:



# each year -- top 10% for 10 years 100 
# 100 3 yr performance: top 25% percentile
# 2016-

## 9. HighCap v2 — gates + momentum overlay

Same cap-rate ranking as v1, but with **hard filter gates** applied first. Counties failing any gate are excluded before ranking. Then within qualified pool, rank by `cap_rate_proxy` (with optional cap-compression momentum tiebreaker).

**Default gates** (edit `GATES` dict below):
- `pop_yoy_pct > 0` — positive population growth
- `unemployment_rate < 6.5%` — labor market health
- `rental_vacancy_rate < 12%` — rental demand exists
- `renter_share > 20%` — meaningful renter population
- `any_disqualifier == FALSE` — passes the mart's built-in gates (aging trap, sector concentration, price runup)

In [103]:
# Load the extra columns we need for gates (plus disqualifier flags)
con_g = duckdb.connect(WAREHOUSE, read_only=True)
gate_ctx = con_g.execute('''
    SELECT fips,
           EXTRACT(YEAR FROM period_end_date)::INT AS year,
           unemployment_rate,
           rental_vacancy_rate,
           renter_share,
           mf_cap_rate_change_3yr,
           any_disqualifier
    FROM main_marts.mart_county_features
    WHERE EXTRACT(MONTH FROM period_end_date) = 12
''').df()
con_g.close()

# GATES — edit thresholds here
GATES = {
    "min_pop_yoy_pct":       0.0,     # pop growing (>= 0%)
    "max_unemp_pct":         6.5,     # unemployment below 6.5% (mart stores as percent, e.g. 4.8)
    "max_vacancy_frac":      0.12,    # rental vacancy below 12% (mart stores as fraction)
    "min_renter_share_frac": 0.20,    # at least 20% renters
    "require_no_disq":       True,    # skip counties flagged by any mart disqualifier
}

MOMENTUM_TIEBREAK = True   # if True: prefer cap-COMPRESSING markets (negative 3yr change)

def highcap_v2_picks_for(year, gates=GATES, momentum_tiebreak=MOMENTUM_TIEBREAK):
    '''Gate-filtered HighCap picks. Returns top-N by cap after gates applied.'''
    # Start from universe (uni already has pop, cap, etc.)
    c = uni[uni['year'] == year].dropna(subset=['cap_rate_proxy']).copy()

    # Attach gate columns
    c = c.merge(gate_ctx[gate_ctx['year'] == year][['fips','unemployment_rate','rental_vacancy_rate','renter_share','mf_cap_rate_change_3yr','any_disqualifier']],
                on='fips', how='left')

    # PIT pop YoY
    pop_yr = pop_yoy_cache.get(year, pd.DataFrame())
    if not pop_yr.empty:
        c = c.merge(pop_yr[['fips','pop_yoy_pct']], on='fips', how='left')
    else:
        c['pop_yoy_pct'] = np.nan

    # Apply gates — RELAXED: if data is NaN, skip that gate for that county
    # (i.e., missing data doesn't disqualify — only failing a KNOWN value disqualifies)
    n_before = len(c)
    if gates.get('min_pop_yoy_pct') is not None:
        c = c[c['pop_yoy_pct'].isna() | (c['pop_yoy_pct'] >= gates['min_pop_yoy_pct'])]
    if gates.get('max_unemp_pct') is not None:
        c = c[c['unemployment_rate'].isna() | (c['unemployment_rate'] <= gates['max_unemp_pct'])]
    if gates.get('max_vacancy_frac') is not None:
        c = c[c['rental_vacancy_rate'].isna() | (c['rental_vacancy_rate'] <= gates['max_vacancy_frac'])]
    if gates.get('min_renter_share_frac') is not None:
        c = c[c['renter_share'].isna() | (c['renter_share'] >= gates['min_renter_share_frac'])]
    if gates.get('require_no_disq'):
        c = c[(c['any_disqualifier'] == False) | (c['any_disqualifier'].isna())]
    n_after = len(c)

    if c.empty:
        return pd.DataFrame(), (n_before, n_after)

    # Rank: cap desc, then cap-compression momentum (more negative = better) as tiebreak
    if momentum_tiebreak and 'mf_cap_rate_change_3yr' in c.columns:
        c = c.sort_values(['cap_rate_proxy','mf_cap_rate_change_3yr'], ascending=[False, True])
    else:
        c = c.sort_values('cap_rate_proxy', ascending=False)

    picked = []
    state_counts = {}
    for _, r in c.iterrows():
        if state_counts.get(r['state_name'], 0) >= MAX_PER_STATE:
            continue
        picked.append(r)
        state_counts[r['state_name']] = state_counts.get(r['state_name'], 0) + 1
        if len(picked) >= TOP_N:
            break
    return pd.DataFrame(picked), (n_before, n_after)

# Sanity check
picks_2025, (nb_, na_) = highcap_v2_picks_for(2025)
print(f'2025: {nb_} counties before gates, {na_} after (pass rate {na_/nb_*100:.1f}%)')
print(f'HighCap v2 top-{TOP_N} for 2025:')
print(picks_2025[['county_name','state_name','cap_rate_proxy','unemployment_rate','rental_vacancy_rate','renter_share','pop_yoy_pct']].to_string(index=False))

2025: 87 counties before gates, 62 after (pass rate 71.3%)
HighCap v2 top-5 for 2025:
      county_name     state_name  cap_rate_proxy  unemployment_rate  rental_vacancy_rate  renter_share  pop_yoy_pct
Cumberland County North Carolina        0.072387                4.2             0.058267      0.409052     0.623519
   Lubbock County          Texas        0.065243                3.1             0.094959      0.445668     0.956353
      Lake County        Florida        0.060083                4.8             0.091112      0.231109     2.914891
  Escambia County        Florida        0.059873                5.0             0.082342      0.333777     1.019346
Burlington County     New Jersey        0.059046                4.2             0.029663      0.231698     0.135808


### 9.1 Side-by-side: HC v1 / HC v2 / HC v3 / HC v4 / ELITE — per year + summary

**Prerequisites**: run Section 10 (Cell 34) and Section 12 (Cell 40) first so `highcap_v3_picks_for` and `highcap_v4_picks_for` are defined.

In [ ]:
# GUARDS: define highcap_v3_picks_for and highcap_v4_picks_for inline if
# Sections 10 and 12 haven't been run yet. Makes this cell self-contained.
if 'highcap_v3_picks_for' not in dir():
    print('  Loading HC v3 helper inline...')
    _con_v3 = duckdb.connect(WAREHOUSE, read_only=True)
    _v3_ctx = _con_v3.execute('''
        SELECT fips, EXTRACT(YEAR FROM period_end_date)::INT AS year,
               hpi_trailing_24mo, permits_intensity_ratio,
               unemployment_rate_yoy_chg, any_disqualifier
        FROM main_marts.mart_county_features
        WHERE EXTRACT(MONTH FROM period_end_date) = 12
    ''').df()
    _con_v3.close()
    _GATES_v3 = {'max_hpi_24mo': 0.30, 'max_permit_ratio': 1.5,
                 'max_unemp_yoy': 1.0, 'require_no_disq': True}
    def highcap_v3_picks_for(year, gates=_GATES_v3):
        c = uni[uni['year'] == year].dropna(subset=['cap_rate_proxy']).copy()
        c = c.merge(_v3_ctx[_v3_ctx['year'] == year][['fips','hpi_trailing_24mo','permits_intensity_ratio','unemployment_rate_yoy_chg','any_disqualifier']], on='fips', how='left')
        n_before = len(c)
        c = c[c['hpi_trailing_24mo'].isna() | (c['hpi_trailing_24mo'] < gates['max_hpi_24mo'])]
        c = c[c['permits_intensity_ratio'].isna() | (c['permits_intensity_ratio'] < gates['max_permit_ratio'])]
        c = c[c['unemployment_rate_yoy_chg'].isna() | (c['unemployment_rate_yoy_chg'] < gates['max_unemp_yoy'])]
        if gates['require_no_disq']:
            c = c[(c['any_disqualifier'] == False) | (c['any_disqualifier'].isna())]
        n_after = len(c)
        if c.empty: return pd.DataFrame(), (n_before, n_after)
        c = c.sort_values('cap_rate_proxy', ascending=False)
        picked, sc = [], {}
        for _, r in c.iterrows():
            if sc.get(r['state_name'], 0) >= MAX_PER_STATE: continue
            picked.append(r); sc[r['state_name']] = sc.get(r['state_name'], 0) + 1
            if len(picked) >= TOP_N: break
        return pd.DataFrame(picked), (n_before, n_after)

if 'highcap_v4_picks_for' not in dir():
    print('  Loading HC v4 helper inline...')
    _con_v4 = duckdb.connect(WAREHOUSE, read_only=True)
    _zori_yoy_df = _con_v4.execute('''
        SELECT fips, EXTRACT(YEAR FROM period_end_date)::INT AS year, zori_yoy
        FROM main_marts.mart_county_features
        WHERE EXTRACT(MONTH FROM period_end_date) = 12
    ''').df()
    _con_v4.close()
    def highcap_v4_picks_for(year, momentum_col='zori_yoy', shortlist_mult=3):
        c = uni[uni['year'] == year].dropna(subset=['cap_rate_proxy']).copy()
        c = c.merge(_zori_yoy_df[_zori_yoy_df['year'] == year][['fips','zori_yoy']], on='fips', how='left')
        K = TOP_N * shortlist_mult
        shortlist = c.nlargest(K, 'cap_rate_proxy').copy()
        shortlist = shortlist.sort_values(momentum_col, ascending=False, na_position='last')
        picked, sc = [], {}
        for _, r in shortlist.iterrows():
            if sc.get(r['state_name'], 0) >= MAX_PER_STATE: continue
            picked.append(r); sc[r['state_name']] = sc.get(r['state_name'], 0) + 1
            if len(picked) >= TOP_N: break
        return pd.DataFrame(picked), len(shortlist)

# Section 9.1 side-by-side: HC v1 / HC v2 / HC v3 / HC v4 / ELITE — per year + summary
# Requires Section 10 (v3) and Section 12 (v4) to have run.
year_tr_lookup = uni[['fips','year','tr_zori36']].dropna(subset=['tr_zori36'])

summary_all = []
for r in elite_results:
    yr = r.test_year
    if yr < 2015:
        continue

    el = r.top5.head(TOP_N)[['fips','county_name','state_name']].copy()
    v1_raw = highcap_picks_for(yr)
    v1 = v1_raw.head(TOP_N)[['fips','county_name','state_name']].copy() if not v1_raw.empty else pd.DataFrame()
    v2_raw, (nb_, na_) = highcap_v2_picks_for(yr)
    v2 = v2_raw.head(TOP_N)[['fips','county_name','state_name']].copy() if not v2_raw.empty else pd.DataFrame()
    v3_raw, (nb_v3, na_v3) = highcap_v3_picks_for(yr)
    v3 = v3_raw.head(TOP_N)[['fips','county_name','state_name']].copy() if not v3_raw.empty else pd.DataFrame()
    v4_raw, _ = highcap_v4_picks_for(yr)
    v4 = v4_raw.head(TOP_N)[['fips','county_name','state_name']].copy() if not v4_raw.empty else pd.DataFrame()

    year_tr = year_tr_lookup[year_tr_lookup['year'] == yr]
    tr_map = dict(zip(year_tr['fips'], year_tr['tr_zori36'])) if len(year_tr) else {}

    def _avg(picks):
        if picks.empty: return np.nan
        trs = [tr_map.get(f) for f in picks['fips'] if tr_map.get(f) is not None]
        return np.nanmean(trs)*100 if trs else np.nan

    tr_v1 = _avg(v1); tr_v2 = _avg(v2); tr_v3 = _avg(v3); tr_v4 = _avg(v4); tr_el = _avg(el)
    uni_tr = year_tr['tr_zori36'].mean()*100 if len(year_tr) else np.nan

    def _s(v): return 'n/a' if pd.isna(v) else f'{v:+.2f}%'

    print('')
    print('=' * 170)
    print(f"YEAR {yr}   uni_n={len(uni[uni['year']==yr])}   v2 gates={na_}/{nb_}   v3 gates={na_v3}/{nb_v3}")
    print(f'    HC_v1={_s(tr_v1)}   HC_v2={_s(tr_v2)}   HC_v3={_s(tr_v3)}   HC_v4={_s(tr_v4)}   ELITE={_s(tr_el)}   Uni={_s(uni_tr)}')
    print('=' * 170)

    def name_of(picks, i):
        if i >= len(picks): return ''
        r = picks.iloc[i]
        return f"{r['county_name'].replace(' County','')}, {r['state_name'][:2]}"

    print(f"{'#':<3}  {'HC_v1 (raw cap)':<22}  {'HC_v2 (grow gates)':<22}  {'HC_v3 (tops guards)':<22}  {'HC_v4 (momentum)':<22}  {'ELITE':<22}")
    for i in range(TOP_N):
        print(f'#{i+1:<2}  {name_of(v1, i):<22}  {name_of(v2, i):<22}  {name_of(v3, i):<22}  {name_of(v4, i):<22}  {name_of(el, i):<22}')

    summary_all.append({
        'year': yr, 'uni_TR': uni_tr,
        'HC_v1_TR': tr_v1, 'HC_v2_TR': tr_v2, 'HC_v3_TR': tr_v3, 'HC_v4_TR': tr_v4, 'ELITE_TR': tr_el,
    })

print()
print('=' * 170)
print('SUMMARY — HC v1 / HC v2 / HC v3 / HC v4 / ELITE (observable cohorts)')
print('=' * 170)
s_df = pd.DataFrame(summary_all)
for col in ['HC_v1','HC_v2','HC_v3','HC_v4','ELITE']:
    s_df[f'{col}_edge'] = s_df[f'{col}_TR'] - s_df['uni_TR']
print(s_df.to_string(index=False, float_format=lambda x: f'{x:+.2f}' if pd.notna(x) else 'n/a'))

obs = s_df.dropna(subset=['HC_v1_TR','HC_v2_TR','HC_v3_TR','HC_v4_TR','ELITE_TR'])
if len(obs):
    print()
    print(f'Aggregate over {len(obs)} observable cohorts:')
    for label, col in [
        ('HC v1 (raw cap)      ', 'HC_v1'),
        ('HC v2 (growth gates) ', 'HC_v2'),
        ('HC v3 (tops guards)  ', 'HC_v3'),
        ('HC v4 (cap+momentum) ', 'HC_v4'),
        ('ELITE                ', 'ELITE'),
    ]:
        avg = obs[f'{col}_TR'].mean()
        edge = obs[f'{col}_edge'].mean()
        wins = (obs[f'{col}_edge'] > 0).sum()
        print(f'  {label} avg TR: {avg:+.2f}%   edge: {edge:+.2f}pp   wins {wins}/{len(obs)}')

### 9.2 Side-by-side picks with ZORI 36mo + TR per pick — all 5 methods

Per-pick view for HC v1 / HC v2 / HC v3 / HC v4 / ELITE. Each pick shows realized `zori_forward_return_36mo` and `tr_zori36`. Cohorts past 2022 show `n/a` since the 3-year horizon isn't observable yet.

**Prerequisites**: same as 9.1 — run Sections 10 and 12 first.

In [ ]:
# GUARDS: define highcap_v3_picks_for and highcap_v4_picks_for inline if
# Sections 10 and 12 haven't been run yet. Makes this cell self-contained.
if 'highcap_v3_picks_for' not in dir():
    print('  Loading HC v3 helper inline...')
    _con_v3 = duckdb.connect(WAREHOUSE, read_only=True)
    _v3_ctx = _con_v3.execute('''
        SELECT fips, EXTRACT(YEAR FROM period_end_date)::INT AS year,
               hpi_trailing_24mo, permits_intensity_ratio,
               unemployment_rate_yoy_chg, any_disqualifier
        FROM main_marts.mart_county_features
        WHERE EXTRACT(MONTH FROM period_end_date) = 12
    ''').df()
    _con_v3.close()
    _GATES_v3 = {'max_hpi_24mo': 0.30, 'max_permit_ratio': 1.5,
                 'max_unemp_yoy': 1.0, 'require_no_disq': True}
    def highcap_v3_picks_for(year, gates=_GATES_v3):
        c = uni[uni['year'] == year].dropna(subset=['cap_rate_proxy']).copy()
        c = c.merge(_v3_ctx[_v3_ctx['year'] == year][['fips','hpi_trailing_24mo','permits_intensity_ratio','unemployment_rate_yoy_chg','any_disqualifier']], on='fips', how='left')
        n_before = len(c)
        c = c[c['hpi_trailing_24mo'].isna() | (c['hpi_trailing_24mo'] < gates['max_hpi_24mo'])]
        c = c[c['permits_intensity_ratio'].isna() | (c['permits_intensity_ratio'] < gates['max_permit_ratio'])]
        c = c[c['unemployment_rate_yoy_chg'].isna() | (c['unemployment_rate_yoy_chg'] < gates['max_unemp_yoy'])]
        if gates['require_no_disq']:
            c = c[(c['any_disqualifier'] == False) | (c['any_disqualifier'].isna())]
        n_after = len(c)
        if c.empty: return pd.DataFrame(), (n_before, n_after)
        c = c.sort_values('cap_rate_proxy', ascending=False)
        picked, sc = [], {}
        for _, r in c.iterrows():
            if sc.get(r['state_name'], 0) >= MAX_PER_STATE: continue
            picked.append(r); sc[r['state_name']] = sc.get(r['state_name'], 0) + 1
            if len(picked) >= TOP_N: break
        return pd.DataFrame(picked), (n_before, n_after)

if 'highcap_v4_picks_for' not in dir():
    print('  Loading HC v4 helper inline...')
    _con_v4 = duckdb.connect(WAREHOUSE, read_only=True)
    _zori_yoy_df = _con_v4.execute('''
        SELECT fips, EXTRACT(YEAR FROM period_end_date)::INT AS year, zori_yoy
        FROM main_marts.mart_county_features
        WHERE EXTRACT(MONTH FROM period_end_date) = 12
    ''').df()
    _con_v4.close()
    def highcap_v4_picks_for(year, momentum_col='zori_yoy', shortlist_mult=3):
        c = uni[uni['year'] == year].dropna(subset=['cap_rate_proxy']).copy()
        c = c.merge(_zori_yoy_df[_zori_yoy_df['year'] == year][['fips','zori_yoy']], on='fips', how='left')
        K = TOP_N * shortlist_mult
        shortlist = c.nlargest(K, 'cap_rate_proxy').copy()
        shortlist = shortlist.sort_values(momentum_col, ascending=False, na_position='last')
        picked, sc = [], {}
        for _, r in shortlist.iterrows():
            if sc.get(r['state_name'], 0) >= MAX_PER_STATE: continue
            picked.append(r); sc[r['state_name']] = sc.get(r['state_name'], 0) + 1
            if len(picked) >= TOP_N: break
        return pd.DataFrame(picked), len(shortlist)

# Section 9.2 side-by-side: HC v1 / HC v2 / HC v3 / HC v4 / ELITE — ZORI36 + TR per pick
# Requires Section 10 (v3) and Section 12 (v4) to have run.
zori_tr_lookup = uni[['fips','year','zori_forward_return_36mo','tr_zori36']].copy()

def _enrich_returns(picks, year):
    if picks.empty: return picks
    return picks[['fips','county_name','state_name']].merge(
        zori_tr_lookup[zori_tr_lookup['year'] == year], on='fips', how='left')

def _fmt_pct(v, w=6):
    return f'{v*100:+{w}.2f}%' if pd.notna(v) else f"{'n/a':>{w+1}s}"

def _cell(df, i, name_w=18):
    if df.empty or i >= len(df):
        return f"{'':<{name_w}} {'':>7} {'':>7}"
    r = df.iloc[i]
    name = f"{r['county_name'].replace(' County','')}, {r['state_name'][:2]}"[:name_w]
    return f"{name:<{name_w}} {_fmt_pct(r.get('zori_forward_return_36mo')):>7} {_fmt_pct(r.get('tr_zori36')):>7}"

def _avg_cell(df, name_w=18):
    if df.empty or df['tr_zori36'].isna().all():
        return f"{'AVG':<{name_w}} {'n/a':>7} {'n/a':>7}"
    az = df['zori_forward_return_36mo'].mean(skipna=True)
    at = df['tr_zori36'].mean(skipna=True)
    return f"{'AVG':<{name_w}} {_fmt_pct(az):>7} {_fmt_pct(at):>7}"

for r in elite_results:
    yr = r.test_year
    if yr < 2015:
        continue

    el = _enrich_returns(r.top5.head(TOP_N)[['fips','county_name','state_name']], yr)
    v1_raw = highcap_picks_for(yr)
    v1 = _enrich_returns(v1_raw.head(TOP_N), yr) if not v1_raw.empty else pd.DataFrame()
    v2_raw, _ = highcap_v2_picks_for(yr)
    v2 = _enrich_returns(v2_raw.head(TOP_N), yr) if not v2_raw.empty else pd.DataFrame()
    v3_raw, _ = highcap_v3_picks_for(yr)
    v3 = _enrich_returns(v3_raw.head(TOP_N), yr) if not v3_raw.empty else pd.DataFrame()
    v4_raw, _ = highcap_v4_picks_for(yr)
    v4 = _enrich_returns(v4_raw.head(TOP_N), yr) if not v4_raw.empty else pd.DataFrame()

    sep = '=' * 210
    print(sep)
    print(f'YEAR {yr}   universe_n={len(uni[uni.year == yr])}')
    print(sep)
    header = (f"{'#':<3}  "
              f"{'HC v1 (raw cap)':<18} {'ZORI':>7} {'TR':>7}   "
              f"{'HC v2 (grow gates)':<18} {'ZORI':>7} {'TR':>7}   "
              f"{'HC v3 (tops guards)':<18} {'ZORI':>7} {'TR':>7}   "
              f"{'HC v4 (momentum)':<18} {'ZORI':>7} {'TR':>7}   "
              f"{'ELITE':<18} {'ZORI':>7} {'TR':>7}")
    print(header)
    print('-' * len(header))
    for i in range(TOP_N):
        print(f'#{i+1:<2}  {_cell(v1, i)}   {_cell(v2, i)}   {_cell(v3, i)}   {_cell(v4, i)}   {_cell(el, i)}')
    print(f'    {_avg_cell(v1)}   {_avg_cell(v2)}   {_avg_cell(v3)}   {_avg_cell(v4)}   {_avg_cell(el)}')
    print()

## 10. HighCap v3 — "guard against tops" gates

Different philosophy from v2: instead of filtering out structurally weak markets (which turned out to hurt), filter out markets showing signs of being at a **cyclical top** or in early recession.

**GATES_v3:**
- `hpi_trailing_24mo < 0.30` — hasn't run up 30% in 2 years (top-tick guard)
- `permits_intensity_ratio < 1.5` — not oversupplied (current permits vs 10yr baseline)
- `unemployment_rate_yoy_chg < 1.0` — unemployment not spiking (recession warning)
- `any_disqualifier == FALSE` — mart's built-in structural filters (aging trap, sector concentration, HPI runup)

Aim: keep the "distressed high-cap mean-reversion" plays but exclude "high-cap because we already ran the market up and it's about to correct" plays.

In [ ]:
# Load the additional gate columns
con_g3 = duckdb.connect(WAREHOUSE, read_only=True)
gate_v3_ctx = con_g3.execute('''
    SELECT fips,
           EXTRACT(YEAR FROM period_end_date)::INT AS year,
           hpi_trailing_24mo,
           permits_intensity_ratio,
           unemployment_rate_yoy_chg,
           any_disqualifier
    FROM main_marts.mart_county_features
    WHERE EXTRACT(MONTH FROM period_end_date) = 12
''').df()
con_g3.close()

GATES_v3 = {
    "max_hpi_24mo_runup":   0.30,   # HPI hasn't run 30%+ in 24mo (top-tick guard)
    "max_permit_intensity": 1.5,    # permits <=1.5x 10yr baseline (supply guard)
    "max_unemp_yoy_chg":    1.0,    # unemployment change <= +1pp (recession guard)
    "require_no_disq":      True,   # mart's built-in disqualifiers
}

def highcap_v3_picks_for(year, gates=GATES_v3, momentum_tiebreak=True):
    '''Guard-against-tops HighCap. Uses momentum/supply/recession gates rather than fundamental gates.'''
    c = uni[uni['year'] == year].dropna(subset=['cap_rate_proxy']).copy()

    c = c.merge(gate_v3_ctx[gate_v3_ctx['year'] == year][['fips','hpi_trailing_24mo','permits_intensity_ratio','unemployment_rate_yoy_chg','any_disqualifier']],
                on='fips', how='left')
    # Also pull mf_cap_rate_change_3yr from gate_ctx (loaded in Section 9)
    if 'gate_ctx' in globals():
        c = c.merge(gate_ctx[gate_ctx['year'] == year][['fips','mf_cap_rate_change_3yr']], on='fips', how='left')

    n_before = len(c)

    # Apply gates — RELAXED: missing data doesn't disqualify
    if gates.get('max_hpi_24mo_runup') is not None:
        c = c[c['hpi_trailing_24mo'].isna() | (c['hpi_trailing_24mo'] <= gates['max_hpi_24mo_runup'])]
    if gates.get('max_permit_intensity') is not None:
        c = c[c['permits_intensity_ratio'].isna() | (c['permits_intensity_ratio'] <= gates['max_permit_intensity'])]
    if gates.get('max_unemp_yoy_chg') is not None:
        c = c[c['unemployment_rate_yoy_chg'].isna() | (c['unemployment_rate_yoy_chg'] <= gates['max_unemp_yoy_chg'])]
    if gates.get('require_no_disq'):
        c = c[(c['any_disqualifier'] == False) | (c['any_disqualifier'].isna())]

    n_after = len(c)
    if c.empty:
        return pd.DataFrame(), (n_before, n_after)

    # Rank
    if momentum_tiebreak and 'mf_cap_rate_change_3yr' in c.columns:
        c = c.sort_values(['cap_rate_proxy','mf_cap_rate_change_3yr'], ascending=[False, True])
    else:
        c = c.sort_values('cap_rate_proxy', ascending=False)

    picked, sc = [], {}
    for _, r in c.iterrows():
        if sc.get(r['state_name'], 0) >= MAX_PER_STATE: continue
        picked.append(r); sc[r['state_name']] = sc.get(r['state_name'], 0) + 1
        if len(picked) >= TOP_N: break
    return pd.DataFrame(picked), (n_before, n_after)

# Sanity check
picks_25_v3, (nb_, na_) = highcap_v3_picks_for(2025)
print(f'2025: {nb_} → {na_} counties after v3 gates ({na_/nb_*100:.0f}% pass rate)')
print(f'HighCap v3 top-{TOP_N} for 2025:')
print(picks_25_v3[['county_name','state_name','cap_rate_proxy','hpi_trailing_24mo','permits_intensity_ratio','unemployment_rate_yoy_chg']].to_string(index=False))

### 10.1 4-way side-by-side per year — HC v1 vs HC v2 vs HC v3 vs ELITE

In [ ]:
summary_v3 = []
for r in elite_results:
    yr = r.test_year
    if yr < 2015: continue

    el_top = r.top5.head(TOP_N)[["fips","county_name","state_name"]].copy()
    hc_v1 = highcap_picks_for(yr).head(TOP_N)[["fips","county_name","state_name"]].copy()
    hc_v2, _ = highcap_v2_picks_for(yr)
    hc_v2 = hc_v2.head(TOP_N)[["fips","county_name","state_name"]].copy() if not hc_v2.empty else pd.DataFrame()
    hc_v3, (nb_v3, na_v3) = highcap_v3_picks_for(yr)
    hc_v3 = hc_v3.head(TOP_N)[["fips","county_name","state_name"]].copy() if not hc_v3.empty else pd.DataFrame()

    year_tr = year_tr_lookup[year_tr_lookup['year'] == yr]
    tr_map = dict(zip(year_tr['fips'], year_tr['tr_zori36'])) if len(year_tr) else {}
    def avg_tr(picks):
        if picks.empty: return np.nan
        trs = [tr_map.get(f) for f in picks['fips'] if tr_map.get(f) is not None]
        return np.nanmean(trs)*100 if trs else np.nan

    v1_tr = avg_tr(hc_v1); v2_tr = avg_tr(hc_v2); v3_tr = avg_tr(hc_v3); el_tr_ = avg_tr(el_top)
    uni_tr = year_tr['tr_zori36'].mean()*100 if len(year_tr) else np.nan

    def fmt(v): return "n/a" if pd.isna(v) else f"{v:+.2f}%"
    print("")
    print("=" * 130)
    print(f"YEAR {yr}   uni_n={len(uni[uni['year']==yr])}   v3 gate pass={na_v3}/{nb_v3} ({na_v3/nb_v3*100:.0f}%)")
    print(f"  HC_v1={fmt(v1_tr)}   HC_v2={fmt(v2_tr)}   HC_v3={fmt(v3_tr)}   ELITE={fmt(el_tr_)}   Universe={fmt(uni_tr)}")
    print("=" * 130)

    def name_of(picks, i):
        if i >= len(picks): return ""
        r = picks.iloc[i]
        return f"{r['county_name'].replace(' County','')} {r['state_name'][:2]}"

    print(f"{'#':<3}  {'HC_v1 (raw)':<22}  {'HC_v2 (grow gates)':<22}  {'HC_v3 (tops gates)':<22}  {'ELITE':<22}")
    for i in range(TOP_N):
        print(f"#{i+1:<2}  {name_of(hc_v1, i):<22}  {name_of(hc_v2, i):<22}  {name_of(hc_v3, i):<22}  {name_of(el_top, i):<22}")

    summary_v3.append({
        'year': yr, 'gate_v3_pass': na_v3,
        'HC_v1_TR': v1_tr, 'HC_v2_TR': v2_tr, 'HC_v3_TR': v3_tr,
        'ELITE_TR': el_tr_, 'uni_TR': uni_tr,
    })

s3 = pd.DataFrame(summary_v3)
for col in ['HC_v1','HC_v2','HC_v3','ELITE']:
    s3[f'{col}_edge'] = s3[f'{col}_TR'] - s3['uni_TR']

print()
print("=" * 130)
print("SUMMARY — HC_v1 vs HC_v2 vs HC_v3 vs ELITE")
print("=" * 130)
disp = s3[['year','gate_v3_pass','HC_v1_TR','HC_v2_TR','HC_v3_TR','ELITE_TR','uni_TR','HC_v1_edge','HC_v2_edge','HC_v3_edge','ELITE_edge']]
print(disp.to_string(index=False, float_format=lambda x: f'{x:+.2f}' if pd.notna(x) else 'n/a'))

obs = s3.dropna(subset=['HC_v1_TR','HC_v2_TR','HC_v3_TR'])
if len(obs):
    print()
    print(f"Aggregate over {len(obs)} observable cohorts:")
    print(f"  HC v1 (raw cap)         avg TR: {obs['HC_v1_TR'].mean():+.2f}%   edge: {obs['HC_v1_edge'].mean():+.2f}pp")
    print(f"  HC v2 (growth gates)    avg TR: {obs['HC_v2_TR'].mean():+.2f}%   edge: {obs['HC_v2_edge'].mean():+.2f}pp")
    print(f"  HC v3 (tops guards)     avg TR: {obs['HC_v3_TR'].mean():+.2f}%   edge: {obs['HC_v3_edge'].mean():+.2f}pp")
    print(f"  ELITE                   avg TR: {obs['ELITE_TR'].mean():+.2f}%   edge: {obs['ELITE_edge'].mean():+.2f}pp")
    v3_vs_v1 = obs['HC_v3_TR'] - obs['HC_v1_TR']
    print(f"  HC v3 - HC v1: {v3_vs_v1.mean():+.2f}pp avg   (v3 wins {(v3_vs_v1>0).sum()}/{len(obs)})")

## 11. HC v1 vs v2 vs v3 — per-pick performance

For each cohort, show all three variants' top-5 picks with rent, cap Δ, and TR side by side. Lets you see which specific picks each variant swapped in/out and their realized outcomes.

In [ ]:
# Helper for per-year comparison — run this once
enrich_all = uni[["fips","year","cap_rate_proxy","cap_exit","zori_forward_return_36mo","tr_zori36"]].copy()
enrich_all = enrich_all.rename(columns={"cap_rate_proxy":"cap_entry"})

def show_year_variants(yr):
    """Print HC v1 / v2 / v3 picks for one cohort year."""
    r = next((rr for rr in elite_results if rr.test_year == yr), None)
    if r is None:
        print(f"No walk-forward result for {yr}")
        return

    hc_v1 = highcap_picks_for(yr).head(TOP_N)
    hc_v2_p, _ = highcap_v2_picks_for(yr)
    hc_v2 = hc_v2_p.head(TOP_N) if not hc_v2_p.empty else pd.DataFrame()
    hc_v3_p, _ = highcap_v3_picks_for(yr)
    hc_v3 = hc_v3_p.head(TOP_N) if not hc_v3_p.empty else pd.DataFrame()

    def enrich(picks):
        if picks.empty: return picks
        return picks[["fips","county_name","state_name"]].merge(enrich_all[enrich_all["year"] == yr], on="fips", how="left")

    v1_e = enrich(hc_v1); v2_e = enrich(hc_v2); v3_e = enrich(hc_v3)

    def fmt_pct(v): return f"{v*100:+6.2f}%" if pd.notna(v) else "   n/a "
    def fmt_bps(entry, exit_):
        if pd.notna(entry) and pd.notna(exit_): return f"{(exit_-entry)*10000:+5.0f}bps"
        return "     n/a"

    print("=" * 90)
    print(f"YEAR {yr}")
    print("=" * 90)

    def print_variant(label, picks):
        print(f"\n{label}")
        print(f"{'#':<3} {'Market':<28} {'cap_in':>7} {'cap_out':>7} {'Δbps':>9} {'ZORI36':>8} {'TR':>8}")
        print("-" * 78)
        for i in range(TOP_N):
            r = picks.iloc[i] if i < len(picks) else None
            if r is None:
                print(f"#{i+1:<2}"); continue
            name = f"{r['county_name'].replace(' County','')}, {r['state_name'][:2]}"
            print(f"#{i+1:<2} {name:<28} {fmt_pct(r['cap_entry'])} {fmt_pct(r['cap_exit'])} {fmt_bps(r['cap_entry'], r['cap_exit'])} {fmt_pct(r['zori_forward_return_36mo'])} {fmt_pct(r['tr_zori36'])}")
        if not picks.empty and picks['tr_zori36'].notna().any():
            avg_tr = picks['tr_zori36'].mean(skipna=True)*100
            avg_zori = picks['zori_forward_return_36mo'].mean(skipna=True)*100
            avg_cap_d = (picks['cap_exit'] - picks['cap_entry']).mean(skipna=True)*10000
            print(f"    AVG:                                              {avg_cap_d:>+5.0f}bps {avg_zori:>+7.2f}% {avg_tr:>+7.2f}%")

    print_variant("HC v1 (raw cap):", v1_e)
    print_variant("HC v2 (growth gates):", v2_e)
    print_variant("HC v3 (tops guards):", v3_e)
    print()

print("Helper loaded. Run individual year cells below.")

## 12. HighCap v4 — cap shortlist + momentum re-rank

**Two-stage sort:**
1. Take top-K counties by `cap_rate_proxy` at entry (default K = 3 × TOP_N = 15)
2. Within that shortlist, re-rank by `zori_yoy` (last-12mo rent growth) descending
3. Take top-N with state cap

Combines the yield-first thesis of HC v1 with a short-horizon rent-momentum tiebreaker.

In [ ]:
# Load zori_yoy from mart (rent momentum at time of prediction)
con_v4 = duckdb.connect(WAREHOUSE, read_only=True)
zori_yoy_df = con_v4.execute('''
    SELECT fips,
           EXTRACT(YEAR FROM period_end_date)::INT AS year,
           zori_yoy,
           hpi_yoy
    FROM main_marts.mart_county_features
    WHERE EXTRACT(MONTH FROM period_end_date) = 12
''').df()
con_v4.close()

CAP_SHORTLIST_MULT = 3   # top K = TOP_N * CAP_SHORTLIST_MULT by cap, then re-rank
MOMENTUM_COL = 'zori_yoy'   # try 'hpi_yoy' for price momentum instead

def highcap_v4_picks_for(year, momentum_col=MOMENTUM_COL, shortlist_mult=CAP_SHORTLIST_MULT):
    '''Two-stage: top-K by cap → re-rank by momentum → top-N with state cap.'''
    c = uni[uni['year'] == year].dropna(subset=['cap_rate_proxy']).copy()
    c = c.merge(zori_yoy_df[zori_yoy_df['year'] == year][['fips','zori_yoy','hpi_yoy']], on='fips', how='left')

    # Stage 1: top-K by cap
    K = TOP_N * shortlist_mult
    shortlist = c.nlargest(K, 'cap_rate_proxy').copy()

    # Stage 2: re-rank by momentum (missing values sorted to bottom)
    shortlist = shortlist.sort_values(momentum_col, ascending=False, na_position='last')

    # Apply state cap
    picked, sc = [], {}
    for _, r in shortlist.iterrows():
        if sc.get(r['state_name'], 0) >= MAX_PER_STATE: continue
        picked.append(r); sc[r['state_name']] = sc.get(r['state_name'], 0) + 1
        if len(picked) >= TOP_N: break
    return pd.DataFrame(picked), len(shortlist)

# Sanity check for 2025
picks_25_v4, sl_size = highcap_v4_picks_for(2025)
print(f'2025 v4 (top-{TOP_N*CAP_SHORTLIST_MULT} by cap, re-rank by {MOMENTUM_COL}): shortlist={sl_size}')
print(picks_25_v4[['county_name','state_name','cap_rate_proxy','zori_yoy','hpi_yoy']].to_string(index=False))

### 12.1 HC v1 vs HC v4 head-to-head per year

In [ ]:
summary_v4 = []
for r in elite_results:
    yr = r.test_year
    if yr < 2015: continue

    hc_v1 = highcap_picks_for(yr).head(TOP_N)
    hc_v4_p, _ = highcap_v4_picks_for(yr)
    hc_v4 = hc_v4_p.head(TOP_N) if not hc_v4_p.empty else pd.DataFrame()

    year_tr = year_tr_lookup[year_tr_lookup['year'] == yr]
    tr_map = dict(zip(year_tr['fips'], year_tr['tr_zori36'])) if len(year_tr) else {}
    def avg_tr(picks):
        if picks.empty: return np.nan
        trs = [tr_map.get(f) for f in picks['fips'] if tr_map.get(f) is not None]
        return np.nanmean(trs)*100 if trs else np.nan

    v1_tr = avg_tr(hc_v1); v4_tr = avg_tr(hc_v4)
    uni_tr = year_tr['tr_zori36'].mean()*100 if len(year_tr) else np.nan

    def fmt(v): return "n/a" if pd.isna(v) else f"{v:+.2f}%"
    print("")
    print("=" * 100)
    print(f"YEAR {yr}   HC_v1={fmt(v1_tr)}   HC_v4={fmt(v4_tr)}   Universe={fmt(uni_tr)}   Δ={fmt(v4_tr - v1_tr) if pd.notna(v1_tr) and pd.notna(v4_tr) else 'n/a'}")
    print("=" * 100)

    def name_of(picks, i):
        if i >= len(picks): return ""
        r = picks.iloc[i]
        return f"{r['county_name'].replace(' County','')} {r['state_name'][:2]}"

    print(f"{'#':<3}  {'HC_v1 (cap only)':<24}  {'HC_v4 (cap + momentum)':<24}")
    for i in range(TOP_N):
        print(f"#{i+1:<2}  {name_of(hc_v1, i):<24}  {name_of(hc_v4, i):<24}")

    summary_v4.append({'year': yr, 'HC_v1_TR': v1_tr, 'HC_v4_TR': v4_tr, 'uni_TR': uni_tr})

s4 = pd.DataFrame(summary_v4)
s4['v4_vs_v1'] = s4['HC_v4_TR'] - s4['HC_v1_TR']
s4['HC_v1_edge'] = s4['HC_v1_TR'] - s4['uni_TR']
s4['HC_v4_edge'] = s4['HC_v4_TR'] - s4['uni_TR']

print()
print("=" * 100)
print("SUMMARY — HC v1 vs HC v4 (cap+momentum)")
print("=" * 100)
print(s4[['year','HC_v1_TR','HC_v4_TR','uni_TR','HC_v1_edge','HC_v4_edge','v4_vs_v1']].to_string(index=False, float_format=lambda x: f'{x:+.2f}' if pd.notna(x) else 'n/a'))

obs = s4.dropna(subset=['HC_v1_TR','HC_v4_TR'])
if len(obs):
    print()
    print(f"Aggregate over {len(obs)} observable cohorts:")
    print(f"  HC v1 avg TR: {obs['HC_v1_TR'].mean():+.2f}%   edge: {obs['HC_v1_edge'].mean():+.2f}pp")
    print(f"  HC v4 avg TR: {obs['HC_v4_TR'].mean():+.2f}%   edge: {obs['HC_v4_edge'].mean():+.2f}pp")
    print(f"  v4 vs v1: {obs['v4_vs_v1'].mean():+.2f}pp avg    v4 wins {(obs['v4_vs_v1']>0).sum()}/{len(obs)} cohorts")

## 13. HC v1 vs HC v4 per-pick performance — all cohorts

For each cohort year, show both methods' top-5 with individual pick rent / cap Δ / TR. Split into one cell per year for readability (no in-cell scrolling).

In [ ]:
# Helper — run once
def show_v1_vs_v4(yr):
    """Print HC v1 and HC v4 top-5 picks + realized performance for one cohort year."""
    hc_v1 = highcap_picks_for(yr).head(TOP_N)
    hc_v4_p, _ = highcap_v4_picks_for(yr)
    hc_v4 = hc_v4_p.head(TOP_N) if not hc_v4_p.empty else pd.DataFrame()

    def enrich(picks):
        if picks.empty: return picks
        return picks[["fips","county_name","state_name"]].merge(enrich_all[enrich_all["year"] == yr], on="fips", how="left")

    v1_e = enrich(hc_v1); v4_e = enrich(hc_v4)

    def fmt_pct(v): return f"{v*100:+6.2f}%" if pd.notna(v) else "   n/a "
    def fmt_bps(entry, exit_):
        if pd.notna(entry) and pd.notna(exit_): return f"{(exit_-entry)*10000:+5.0f}bps"
        return "     n/a"

    # Aggregates
    def avg_stats(picks):
        if picks.empty or picks['tr_zori36'].isna().all(): return None
        return {
            'tr': picks['tr_zori36'].mean(skipna=True)*100,
            'zori': picks['zori_forward_return_36mo'].mean(skipna=True)*100,
            'cap_d': (picks['cap_exit'] - picks['cap_entry']).mean(skipna=True)*10000,
        }
    v1_agg = avg_stats(v1_e); v4_agg = avg_stats(v4_e)

    delta_str = ""
    if v1_agg and v4_agg:
        delta_str = f"   Δ(v4-v1)={v4_agg['tr']-v1_agg['tr']:+.2f}pp"

    print("=" * 100)
    print(f"YEAR {yr}{delta_str}")
    print("=" * 100)

    def print_variant(label, picks, agg):
        print(f"\n{label}")
        print(f"{'#':<3} {'Market':<28} {'cap_in':>7} {'cap_out':>7} {'Δbps':>9} {'ZORI36':>8} {'TR':>8}")
        print("-" * 78)
        for i in range(TOP_N):
            r = picks.iloc[i] if i < len(picks) else None
            if r is None:
                print(f"#{i+1:<2}"); continue
            name = f"{r['county_name'].replace(' County','')}, {r['state_name'][:2]}"
            print(f"#{i+1:<2} {name:<28} {fmt_pct(r['cap_entry'])} {fmt_pct(r['cap_exit'])} {fmt_bps(r['cap_entry'], r['cap_exit'])} {fmt_pct(r['zori_forward_return_36mo'])} {fmt_pct(r['tr_zori36'])}")
        if agg:
            print(f"    AVG:                                              {agg['cap_d']:>+5.0f}bps {agg['zori']:>+7.2f}% {agg['tr']:>+7.2f}%")

    print_variant("HC v1 (cap only):", v1_e, v1_agg)
    print_variant("HC v4 (cap + momentum):", v4_e, v4_agg)
    print()

print("Helper loaded. Run per-year cells below.")